# Supplementary notebook 2. From the lattice of paper 2 into the region of paper 1 (Figs. S3 and S4, Table S8)

Reproduces SI Figs. S3 and S4 and SI Table S8. Reads no data. Run time about 7 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False
T_RUN = 3000      # the run read by Fig. 4 and the copying analysis of Fig. S2
T_POP = 5000      # the four conditions, the hypothesis tests, the sweeps and Fig. 5; the same seeds,
                  # so the first 3,000 steps of these runs are the runs of Fig. 4

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from math import log2

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

CEIL = log2(100)          # 6.6439 bits, the even-coupling ceiling at N = 100
BA_KAPPA = 10.10          # matched scale-free ensemble, control-entropy paper Table 1
BA_DH_DEG = 0.357         # its degree-weighted compression
BA_RDEG = -0.17           # its degree assortativity
N = 100
COL = {'baseline': '#7A7A7A', 'explore': '#2E7D4E', 'imitate': '#2E5090', 'both': '#A4161A'}
CONDS = {'baseline': dict(pr=0.0, pe=0.0), 'explore': dict(pr=0.0, pe=0.05),
         'imitate': dict(pr=0.3, pe=0.0), 'both': dict(pr=0.3, pe=0.05)}

def seeds(n):
    return list(range(n if not QUICK else max(2, n // 5)))

print('ceiling log2 N =', round(CEIL, 4), 'bits')

ceiling log2 N = 6.6439 bits


## Step 0. The model

In [2]:
import numpy as np
import networkx as nx

LOG2 = np.log(2.0)


# ----------------------------------------------------------------------
# measures (definitions of the second study)
# ----------------------------------------------------------------------
def control_entropy(v):
    """Control entropy in bits of a non-negative vector of shares (Eq. 11)."""
    v = np.asarray(v, float)
    tot = v.sum()
    if tot <= 0:
        return 0.0
    p = v / tot
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def eigen_shares(W):
    """Shares from the leading eigenvector of a symmetric coupling matrix."""
    vals, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[:, -1])
    return v / v.sum()


def adjacency_measures(A, W):
    """Measures read from the unweighted structure, computed as in Table 1 of the
    control-entropy paper: eigenvector weighting on the largest connected component,
    degree assortativity on the whole graph."""
    G = nx.from_numpy_array(A.astype(int))
    deg = A.sum(axis=1)
    s = W.sum(axis=1)
    iu, ju = np.nonzero(np.triu(A, 1))

    def edge_corr(v):
        x, y = np.concatenate([v[iu], v[ju]]), np.concatenate([v[ju], v[iu]])
        return float(np.corrcoef(x, y)[0, 1]) if x.std() > 1e-12 else float('nan')

    r_deg, r_str = edge_corr(deg.astype(float)), edge_corr(s)
    if not nx.is_connected(G):
        G = G.subgraph(max(nx.connected_components(G), key=len)).copy()
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()]))
    return dict(H_adj=control_entropy(v), r_deg=float(r_deg), r_str=r_str, gc=G.number_of_nodes())


# ----------------------------------------------------------------------
# the five rules, one function each
# ----------------------------------------------------------------------
def ring_lattice(N, k):
    """Regular ring lattice: every element coupled to its k nearest neighbours."""
    return nx.to_numpy_array(nx.watts_strogatz_graph(N, k, 0.0), dtype=bool)


def softmax_allocation(A, beta, shat):
    """Eq 1. Element i splits one unit of coupling over its partners j with weights
    exp(beta_i * shat_j). beta_i = 0 gives even coupling."""
    L = np.where(A, beta[:, None] * shat[None, :], -np.inf)
    L = L - L.max(axis=1, keepdims=True)
    E = np.where(A, np.exp(L), 0.0)
    return E / E.sum(axis=1, keepdims=True)


def commitment(phi0, phi1, C, fm):
    """Eq 2. Commitment of an element, in the linear form of the published model,
    f = phi0 + phi1 * C, truncated to [0, 1 - fm]."""
    return np.clip(phi0 + phi1 * C, 0.0, 1.0 - fm)


def protection(f, c, pmax, chalf):
    """Eq 8. Saturating protection bought by committed capability f * c."""
    x = f * c
    return np.where(x > 0, pmax / (1.0 + chalf / np.maximum(x, 1e-12)), 0.0)


def fermi(dc, s):
    """Eq 10. Probability of copying a partner whose capability is higher by dc."""
    return 1.0 / (1.0 + np.exp(-s * dc))


def logistic(z, g, theta):
    """Response function of an element, used inside the spread (Eq 5)."""
    return 1.0 / (1.0 + np.exp(-g * (z - theta)))


class ElementModel:
    """Network-agent model of element-level exploration and imitation.

    Each element holds
      beta_i        direction of coupling (0 = even coupling, > 0 = toward stronger partners)
      phi0_i, phi1_i commitment strategy, giving f_i = phi0_i + phi1_i * C_i
      c_i           capability, the accumulated stimulation it compares with partners
    Each coupling (i, j) holds a weight w_ij.
    """

    def __init__(self, N=100, k=6, seed=0,
                 pr=0.3, pe=0.05, sigma=0.1, sigma_f=0.01, s=2.0,   # strategy dynamics
                 M=10, rho=0.5, nE=5, switch=None, psi=0.3,        # feedback, two channels
                 gamma=1.0, pn=0.02, pl=0.3, lam=1.0, g=4.0, theta=1.0,   # seeding and spread
                 fm=0.1, pmax=1.0, chalf=0.1, f0=0.1,              # capability and protection
                 eta=0.05, delta=0.01, w0=1.0,                     # consolidation
                 pw=2.0, kmin=2, loss=1.0,                         # pruning and its cost
                 spread=True, two_part=True):
        self.p = dict(N=N, k=k, pr=pr, pe=pe, sigma=sigma, sigma_f=sigma_f, s=s, M=M, rho=rho,
                      nE=nE, switch=switch, psi=psi, gamma=gamma, pn=pn, pl=pl, lam=lam, g=g,
                      theta=theta, fm=fm, pmax=pmax, chalf=chalf, f0=f0, eta=eta, delta=delta,
                      w0=w0, pw=pw, kmin=kmin, loss=loss, spread=spread, two_part=two_part)
        self.rng = np.random.default_rng(seed)
        self.N, self.k = N, k
        self.A = ring_lattice(N, k)
        self.W = self.A.astype(float) * w0
        self.beta = np.zeros(N)
        self.phi0 = np.full(N, f0)
        self.phi1 = np.zeros(N)
        self.c = np.ones(N)
        self.x = np.zeros(N)          # activation of the previous step
        self.v = np.ones(N) / np.sqrt(N)   # leading eigenvector, kept warm
        self.t = 0
        self.n_act = 0.0
        self.n_rounds = 0.0
        self.set_essential(self.rng.choice(N, nE, replace=False))

    # ---------- state helpers ----------
    def set_essential(self, E):
        self.E = np.asarray(E)
        self.isE = np.zeros(self.N, bool)
        self.isE[self.E] = True

    def strengths(self):
        return self.W.sum(axis=1)

    def shat(self):
        s = self.strengths()
        return s / s.max()

    def centrality(self, iters=8):
        """Leading eigenvector of W by power iteration, warm started, scaled to a
        maximum of one. This is the C of the commitment rule (Eq 2)."""
        v = self.v
        for _ in range(iters):
            v = self.W @ v
            n = np.linalg.norm(v)
            if n < 1e-12:
                v = np.ones(self.N) / np.sqrt(self.N)
                break
            v = v / n
        self.v = np.abs(v)
        return self.v / max(self.v.max(), 1e-12)

    def allocation(self):
        return softmax_allocation(self.A, self.beta, self.shat())

    # ---------- one step ----------
    def feedback(self):
        """Eq 3. Two channels of information.
        Process feedback is intrinsic: every element receives afferent input in
        proportion to its own activation in the previous step.
        Result feedback is augmented: M packets, each landing on a task-essential
        element with probability rho and on any element otherwise."""
        p, rng = self.p, self.rng
        u = p['psi'] * self.x.copy()
        if p['M'] > 0:
            to_E = rng.random(p['M']) < p['rho']
            tgt = np.where(to_E, rng.choice(self.E, p['M']), rng.integers(0, self.N, p['M']))
            u = u + np.bincount(tgt, minlength=self.N).astype(float)
        return u

    def spread(self, u, a):
        """Eq 4 and Eq 5. Stimulus is seeded where information arrives and then
        travels along the couplings.
        Seeding: element i is stimulated with probability 1 - (1 - p_n) exp(-gamma u_i),
        so an element fires spontaneously at rate p_n and more often when information
        reaches it.
        Spread: an element already stimulated passes activation to partner j with
        the coupling it allocates, and j responds with probability
        pl * logistic(g (d_j - theta)). Each element activates at most once per step.
        The cascade is run to completion inside the step."""
        p, rng, N = self.p, self.rng, self.N
        act = rng.random(N) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))   # Eq 4
        if not p['spread']:
            self.n_act, self.n_rounds = float(act.sum()), 0.0
            return act
        frontier = act.copy()
        rounds = 0
        while frontier.any() and rounds < 20:
            # drive reaching each candidate through the allocation of active elements
            d = p['lam'] * self.k * (frontier @ a)
            prob = p['pl'] * logistic(d, p['g'], p['theta'])       # Eq 5
            new = (rng.random(N) < prob) & ~act
            act |= new
            frontier = new
            rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

    def step(self):
        p, rng, N = self.p, self.rng, self.N
        if p['switch'] and self.t > 0 and self.t % p['switch'] == 0:
            self.set_essential(rng.choice(N, p['nE'], replace=False))

        C = self.centrality()
        a = self.allocation()                                     # Eq 1
        f = commitment(self.phi0, self.phi1, C, p['fm']) if p['two_part'] else np.zeros(N)  # Eq 2
        u = self.feedback()                                       # Eq 3
        act = self.spread(u, a)                                   # Eq 4, Eq 5
        x = act.astype(float)
        self.x = x

        # Eq 6. Payoff is the stimulation the element gathers. It reaches partners
        # through its own allocation and it integrates in proportion to the coupling
        # it holds, so both the direction and the amount of coupling enter.
        s = self.strengths()
        pay = (s / s.mean()) * (a @ x)
        self.c = pay + np.clip(1.0 - p['fm'] - f, 0.0, None) * self.c   # Eq 7

        # Eq 8 and Eq 9. Commitment protects a coupling against decay.
        P = protection(f, self.c, p['pmax'], p['chalf'])
        Pij = 0.5 * (P[:, None] + P[None, :])
        abar = 0.5 * (a + a.T)
        self.W = np.where(self.A,
                          p['w0'] + (1 - p['delta'] * (1 - Pij)) * (self.W - p['w0'])
                          + p['eta'] * self.k * abar * np.outer(x, x), 0.0)

        # Eq S1. Long-term change: a coupling falls out of use, and unless the
        # capability committed to it holds it, it is lost and replaced elsewhere.
        n_rw = int(p['pw']) + (1 if rng.random() < p['pw'] - int(p['pw']) else 0)
        for _ in range(n_rw):
            self._rewire(P)

        # Eq 10. Imitation along a coupling, synchronous, two independent passes.
        b_old, f0_old, f1_old, c_old = self.beta.copy(), self.phi0.copy(), self.phi1.copy(), self.c.copy()
        for i in np.flatnonzero(rng.random(N) < p['pr']):
            nb = np.flatnonzero(self.A[i])
            j = nb[rng.integers(nb.size)]
            if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                self.beta[i] = b_old[j]
        if p['two_part']:
            for i in np.flatnonzero(rng.random(N) < p['pr']):
                nb = np.flatnonzero(self.A[i])
                j = nb[rng.integers(nb.size)]
                if rng.random() < fermi(c_old[j] - c_old[i], p['s']):
                    self.phi0[i], self.phi1[i] = f0_old[j], f1_old[j]

        # Eq S2. Exploration of both parts of the strategy.
        ex = rng.random(N) < p['pe']
        self.beta[ex] += rng.normal(0.0, p['sigma'], ex.sum())
        if p['two_part']:
            ex0 = rng.random(N) < p['pe']
            self.phi0[ex0] += rng.normal(0.0, p['sigma_f'], ex0.sum())
            ex1 = rng.random(N) < p['pe']
            self.phi1[ex1] += rng.normal(0.0, p['sigma_f'], ex1.sum())
        self.t += 1
        return x

    def _rewire(self, P):
        """One element drops its weakest coupling, unless protection holds it, and
        attaches to a new partner chosen with exp(beta_i * shat_j)."""
        p, rng = self.p, self.rng
        i = rng.integers(self.N)
        deg = self.A.sum(axis=1)
        nb = np.flatnonzero(self.A[i])
        nb = nb[deg[nb] > p['kmin']]
        if deg[i] <= p['kmin'] or nb.size == 0:
            return
        j = nb[np.argmin(self.W[i, nb])]
        if rng.random() < 0.5 * (P[i] + P[j]):     # the coupling is held
            return
        # the route is lost, and with it the capability the two elements had built
        # through it. With loss = 1 this is the capability loss of the published model.
        self.c[i] *= (1.0 - p['loss'])
        self.c[j] *= (1.0 - p['loss'])
        cand = np.flatnonzero(~self.A[i])
        cand = cand[cand != i]
        if cand.size == 0:
            return
        z = self.beta[i] * self.shat()[cand]
        w = np.exp(z - z.max())
        new = rng.choice(cand, p=w / w.sum())
        self.A[i, j] = self.A[j, i] = False
        self.W[i, j] = self.W[j, i] = 0.0
        self.A[i, new] = self.A[new, i] = True
        self.W[i, new] = self.W[new, i] = p['w0']

    # ---------- measures ----------
    def measures(self, adj=False):
        s = self.strengths()
        deg = self.A.sum(axis=1)
        top = np.argsort(s)[-len(self.E):]
        nbE = self.A[self.E].any(axis=0) & ~self.isE
        tot = s.sum()
        C = self.centrality()
        f = commitment(self.phi0, self.phi1, C, self.p['fm'])
        return dict(H_str=control_entropy(s), H_eig=control_entropy(eigen_shares(self.W)),
                    H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                    beta_mean=self.beta.mean(), beta_var=self.beta.var(),
                    f_mean=float(f.mean()), phi0_mean=float(self.phi0.mean()),
                    phi1_mean=float(self.phi1.mean()), c_mean=float(self.c.mean()),
                    act=self.n_act / self.N, rounds=self.n_rounds,
                    hub_align=self.isE[top].mean(),
                    share_E=s[self.isE].sum() / tot, share_nbE=s[nbE].sum() / tot,
                    **(adjacency_measures(self.A, self.W) if adj else {}))

    def run(self, T, every=25, adj=False):
        rec = {}
        for step in range(T):
            self.step()
            if step % every == 0 or step == T - 1:
                m = self.measures(adj=adj)
                m['t'] = self.t
                for key, v in m.items():
                    rec.setdefault(key, []).append(v)
        return {key: np.array(v) for key, v in rec.items()}


def replicate(T, seeds, adj=True, **kw):
    """Run the model for several seeds and return the final measures."""
    out = []
    for sd in seeds:
        m = ElementModel(seed=sd, **kw)
        for _ in range(T):
            m.step()
        out.append(m.measures(adj=adj))
    return {key: np.array([o[key] for o in out]) for key in out[0]}

In [3]:
from scipy.stats import mannwhitneyu

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))

def boot_median(a, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a = np.asarray(a)
    d = [np.median(rng2.choice(a, len(a))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def boot_diff(a, b, reps=5000, seed=0):
    rng2 = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    d = [np.median(rng2.choice(a, len(a))) - np.median(rng2.choice(b, len(b))) for _ in range(reps)]
    return np.percentile(d, [2.5, 97.5])

def summary(tag, v, fmt='.3f'):
    v = np.asarray(v, float); lo, hi = boot_median(v)
    print(f'{tag}: median {np.median(v):{fmt}} [{lo:{fmt}}, {hi:{fmt}}], '
          f'mean {v.mean():{fmt}}, sd {v.std(ddof=1):{fmt}}, range {v.min():{fmt}} to {v.max():{fmt}}')

def compare(tag, a, b, unit=''):
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    d = cliffs_delta(a, b)
    lo, hi = boot_diff(a, b)
    diff = np.median(a) - np.median(b)
    print(f'{tag}: U = {u:.1f}, p = {p:.4f}, Cliff delta = {d:+.2f}, '
          f'median difference = {diff:+.3f}{unit} [{lo:+.3f}, {hi:+.3f}]')
    return dict(U=u, p=p, d=d, diff=diff, lo=lo, hi=hi)

def star(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'

In [4]:
# The second paper (control entropy), Table 1: four topologies at
# N = 100 with exactly 300 links in every realisation, seeds 42 to 141. The generators are those of
# the paper 2 repository. The random ensemble uses the G(n, m) form and the scale-free ensemble
# grows by preferential attachment from a twelve-element ring lattice, which is what holds the
# link count at 300.
N_REAL, SEED0 = 100, 42
def make_ensemble_graph(kind, seed):
    if kind == 'lattice':     return nx.watts_strogatz_graph(N, 6, 0.0, seed=seed)
    if kind == 'small-world': return nx.watts_strogatz_graph(N, 6, 0.1, seed=seed)
    if kind == 'random':      return nx.gnm_random_graph(N, 300, seed=seed)
    if kind == 'scale-free':
        g0 = nx.watts_strogatz_graph(12, 6, 0.0, seed=seed)
        return nx.barabasi_albert_graph(N, 3, seed=seed, initial_graph=g0)
    raise ValueError(kind)

KINDS = ['lattice', 'small-world', 'random', 'scale-free']
KCOL = {'lattice': '#50C878', 'small-world': '#9C27B0', 'random': '#4A90E2', 'scale-free': '#E63946'}

# published values, paper 2 Table 1 (eigenvector weighting on the largest component, degree
# heterogeneity and assortativity on the whole graph) and its degree-weighted column
PUB2 = {'lattice':     dict(H=6.6439, dH=0.000, kappa=6.00,  dH_deg=0.0000, r=float('nan')),
        'small-world': dict(H=6.5832, dH=0.061, kappa=6.09,  dH_deg=0.0115, r=float('nan')),
        'random':      dict(H=6.4820, dH=0.162, kappa=6.90,  dH_deg=0.1126, r=float('nan')),
        'scale-free':  dict(H=6.3176, dH=0.326, kappa=10.10, dH_deg=0.3574, r=-0.17)}
# measured coordinative structures of paper 2, compression below their own ceiling, in bits
MEASURED2 = {'football (11)': 0.072, 'basketball (5)': 0.090, 'cortical (100)': 0.097,
             'handball (6)': 0.302, 'musculoskeletal (270)': 0.340}
# paper 1 (PLOS Computational Biology 2026), Table 3, scale-free ensemble unless stated
PUB1 = dict(kappa_BA=9.67, kappa_ER=6.84, kappa_WS=6.09, gamma=2.78, dhp_BA=3.60, dhp_ER=2.25,
            dhp_WS=1.46, HW_BA=5.5, HW_ER=0.5, HW_WS=0.4, thalf_BA=56, thalf_ER=84, thalf_WS=262)

def ensemble_measures(G):
    A = nx.to_numpy_array(G, dtype=bool)
    W = A.astype(float)
    mm = adjacency_measures(A, W)
    deg = A.sum(1)
    return dict(H_eig=mm['H_adj'], H_deg=control_entropy(deg), kappa=(deg ** 2).mean() / deg.mean(),
                r=mm['r_deg'], A=A)

def regenerate_ensembles(n_real=N_REAL):
    ens = {}
    for kind in KINDS:
        rows = [ensemble_measures(make_ensemble_graph(kind, s)) for s in range(SEED0, SEED0 + n_real)]
        ens[kind] = {k: np.array([r[k] for r in rows]) for k in ('H_eig', 'H_deg', 'kappa', 'r')}
        ens[kind]['example'] = rows[0]['A']
    return ens

## Step 1. The four matched ensembles of paper 2, regenerated

In [5]:
ENS = regenerate_ensembles(N_REAL if not QUICK else 20)
print(f'{len(ENS["lattice"]["kappa"])} realisations per ensemble, 300 links each')
print('ensemble      H eig (pub)        compression (pub)   kappa (pub)      degree-weighted compression (pub)   assortativity (pub)')
REGEN = {}
for kind in KINDS:
    e = ENS[kind]; pub = PUB2[kind]
    dH = CEIL - e['H_eig'].mean(); dHd = CEIL - e['H_deg'].mean()
    r = np.nanmean(e['r'])
    REGEN[kind] = dict(dH=dH, dH_deg=dHd, kappa=e['kappa'].mean(), r=r)
    rs = 'undefined' if np.isnan(r) else f'{r:+.3f}'
    ps = 'not reported' if np.isnan(pub['r']) else f'{pub["r"]:+.2f}'
    print(f'{kind:<12} {e["H_eig"].mean():.4f} ({pub["H"]:.4f})   {max(dH, 0):.3f} ({pub["dH"]:.3f})        '
          f'{e["kappa"].mean():5.2f} ({pub["kappa"]:5.2f})   {max(dHd, 0):.4f} ({pub["dH_deg"]:.4f})                  '
          f'{rs} ({ps})')
ok_regen = all(abs(CEIL - ENS[k]['H_eig'].mean() - PUB2[k]['dH']) < 0.002 for k in KINDS) if not QUICK else True
print('published Table 1 of paper 2 reproduced within 0.002 bits:', ok_regen)

100 realisations per ensemble, 300 links each
ensemble      H eig (pub)        compression (pub)   kappa (pub)      degree-weighted compression (pub)   assortativity (pub)
lattice      6.6439 (6.6439)   0.000 (0.000)         6.00 ( 6.00)   0.0000 (0.0000)                  undefined (not reported)
small-world  6.5832 (6.5832)   0.061 (0.061)         6.09 ( 6.09)   0.0114 (0.0115)                  -0.021 (not reported)
random       6.4820 (6.4820)   0.162 (0.162)         6.90 ( 6.90)   0.1126 (0.1126)                  -0.022 (not reported)
scale-free   6.3176 (6.3176)   0.326 (0.326)        10.10 (10.10)   0.3574 (0.3574)                  -0.171 (-0.17)
published Table 1 of paper 2 reproduced within 0.002 bits: True


## Step 2. Runs from the lattice, recorded along the way

In [6]:
def trace_run(seed=0, T=T_POP, every=25, keep_x=False, **kw):
    m = ElementModel(seed=seed, **kw)
    keep = (0, T // 3, T - 1)
    rec, snaps, X = {}, {}, []
    for t in range(T):
        x = m.step()
        if keep_x:
            X.append(x.astype(np.uint8))
        if t % every == 0 or t == T - 1:
            v = m.v.copy(); mm = m.measures(adj=True); m.v = v     # read without disturbing the run
            mm['t'] = m.t
            for key, v in mm.items():
                rec.setdefault(key, []).append(v)
        if t in keep:
            snaps[t] = (m.A.copy(), m.strengths().copy(), m.isE.copy())
    rec = {key: np.array(v) for key, v in rec.items()}
    rec['share_rest'] = 1 - rec['share_E'] - rec['share_nbE']
    return m, rec, snaps, (np.array(X) if keep_x else None)

NS = seeds(30)
RUNS, RECS, SNAPS, XS = {}, {}, {}, {}
for name in ('both', 'baseline'):
    RUNS[name], RECS[name], SNAPS[name], XS[name] = [], [], [], []
    for sd in NS:
        m, rec, sn, X = trace_run(seed=sd, keep_x=(name == 'both'), **CONDS[name])
        RUNS[name].append(m); RECS[name].append(rec); SNAPS[name].append(sn); XS[name].append(X)
print(f'{len(NS)} seeds traced for the full model and for the baseline')

PATH_RUNS = {}
for name, kw in CONDS.items():
    if name in ('both', 'baseline'):
        PATH_RUNS[name] = RECS[name][0]
    else:
        PATH_RUNS[name] = trace_run(seed=0, **kw)[1]
print('coupling budget at the start and at the end of each path from seed 0 (essential, partners, rest)')
for name, r in PATH_RUNS.items():
    print(f'{name:<9} start {r["share_E"][0]:.2f} {r["share_nbE"][0]:.2f} {max(r["share_rest"][0], 0):.2f}   '
          f'end {r["share_E"][-1]:.2f} {r["share_nbE"][-1]:.2f} {max(r["share_rest"][-1], 0):.2f}')

30 seeds traced for the full model and for the baseline
coupling budget at the start and at the end of each path from seed 0 (essential, partners, rest)
baseline  start 0.05 0.27 0.68   end 0.23 0.60 0.18
explore   start 0.05 0.27 0.68   end 0.25 0.55 0.20
imitate   start 0.05 0.27 0.68   end 0.21 0.63 0.16
both      start 0.05 0.27 0.68   end 0.44 0.56 0.00


## Step 3. The learned structure against the signatures of papers 1 and 2

In [7]:
def cascade(A, W, beta, seed_el, rng, rounds=5, per_link=False):
    s = W.sum(1); a = softmax_allocation(A, beta, s / s.max())
    act = np.zeros(N, bool); act[seed_el] = True; frontier = act.copy()
    for _ in range(rounds):
        d = (frontier.astype(float) @ A.astype(float)) if per_link else 1.0 * 6 * (frontier @ a)
        prob = 0.3 * logistic(d, 4.0, 1.0)
        new = (rng.random(N) < prob) & ~act
        act |= new; frontier = new
        if not frontier.any():
            break
    return act.sum()

def cascade_asymmetry(A, W, beta, rng, trials=50, per_link=False):
    deg = A.sum(1); hub = int(np.argmax(deg))
    per = np.flatnonzero(deg < deg.mean())
    per = per if per.size else np.flatnonzero(deg <= deg.mean())
    rh = np.mean([cascade(A, W, beta, hub, rng, per_link=per_link) for _ in range(trials)])
    rp = np.mean([cascade(A, W, beta, int(rng.choice(per)), rng, per_link=per_link) for _ in range(trials)])
    return rh, rp, rh / rp

def weight_hierarchy(A, W):
    deg = A.sum(1); hubs = deg > deg.mean() + 2 * deg.std(); per = deg < deg.mean()
    iu, ju = np.nonzero(np.triu(A, 1)); w = W[iu, ju]
    hh = w[hubs[iu] & hubs[ju]]; pp = w[per[iu] & per[ju]]
    hp = w[(hubs[iu] & per[ju]) | (per[iu] & hubs[ju])]
    return int(hubs.sum()), (hh.mean() if hh.size else np.nan), (hp.mean() if hp.size else np.nan), pp.mean()

def t_half(rec):
    dH = CEIL - rec['H_str']; final = dH[-4:].mean()
    idx = np.flatnonzero(dH >= 0.5 * final)
    return int(rec['t'][idx[0]]) if idx.size else None

rngc = np.random.default_rng(7)
SIG = {}
for name in ('both', 'baseline'):
    rows = []
    for m, rec in zip(RUNS[name], RECS[name]):
        deg = m.A.sum(1); mm = m.measures(adj=True)
        nh, whh, whp, wpp = weight_hierarchy(m.A, m.W)
        rh, rp, ratio = cascade_asymmetry(m.A, m.W, m.beta, rngc)
        rows.append(dict(kappa=mm['kappa'], r=mm['r_deg'], dH_eig=CEIL - mm['H_adj'], dH_deg=CEIL - mm['H_deg'],
                         dH_str=CEIL - mm['H_str'], hubs=nh, HW=whh / wpp, HW_hp=whp / wpp, R_hub=rh, R_per=rp,
                         dhp=ratio, thalf=t_half(rec), align=mm['hub_align'], maxdeg=deg.max()))
    SIG[name] = {k: np.array([r[k] for r in rows], float) for k in rows[0]}
    print(f'--- {name}, {len(rows)} runs')
    summary('degree heterogeneity kappa', SIG[name]['kappa'], '.2f')
    summary('degree assortativity', SIG[name]['r'])
    summary('compression, eigenvector weighting on the unweighted structure (bits)', SIG[name]['dH_eig'])
    summary('compression, degree weighting (bits)', SIG[name]['dH_deg'])
    summary('compression, strength shares (bits)', SIG[name]['dH_str'])
    summary('hubs by the paper 1 criterion', SIG[name]['hubs'], '.1f')
    v = SIG[name]['HW']; ok = ~np.isnan(v)
    summary('weight hierarchy W_hh / W_pp (runs with hub-hub couplings)', v[ok], '.2f')
    print(f'    runs with at least one hub-hub coupling: {int(ok.sum())} of {len(v)}')
    summary('weight ratio W_hp / W_pp', SIG[name]['HW_hp'], '.2f')
    summary('cascade reach from the strongest hub, 5 rounds', SIG[name]['R_hub'], '.1f')
    summary('cascade reach from a peripheral element, 5 rounds', SIG[name]['R_per'], '.1f')
    summary('cascade asymmetry, hub over peripheral', SIG[name]['dhp'])
    summary('steps to half of the final compression', SIG[name]['thalf'], '.0f')
    summary('hub alignment, essential elements among the five strongest', SIG[name]['align'])
    summary('largest degree', SIG[name]['maxdeg'], '.0f')
print('published references. paper 1 scale-free: kappa', PUB1['kappa_BA'], 'cascade asymmetry', PUB1['dhp_BA'],
      'weight hierarchy', PUB1['HW_BA'], 't_half', PUB1['thalf_BA'], 'of 1000.',
      'paper 2 matched scale-free: kappa', PUB2['scale-free']['kappa'], 'compression', PUB2['scale-free']['dH'],
      'degree-weighted', PUB2['scale-free']['dH_deg'], 'assortativity', PUB2['scale-free']['r'])
print('measured structures of paper 2, compression below their own ceiling (bits):',
      ', '.join(f'{k} {v:.3f}' for k, v in MEASURED2.items()))

--- both, 30 runs
degree heterogeneity kappa: median 21.04 [17.44, 22.99], mean 18.85, sd 6.49, range 6.34 to 28.48
degree assortativity: median -0.299 [-0.316, -0.262], mean -0.264, sd 0.120, range -0.427 to 0.056
compression, eigenvector weighting on the unweighted structure (bits): median 0.245 [0.231, 0.254], mean 0.241, sd 0.060, range 0.078 to 0.378
compression, degree weighting (bits): median 0.709 [0.591, 0.768], mean 0.615, sd 0.248, range 0.041 to 0.925
compression, strength shares (bits): median 1.036 [0.849, 1.164], mean 0.950, sd 0.452, range 0.021 to 1.700
hubs by the paper 1 criterion: median 3.0 [2.0, 3.0], mean 2.9, sd 1.2, range 1.0 to 6.0
weight hierarchy W_hh / W_pp (runs with hub-hub couplings): median 2.14 [1.82, 4.02], mean 4.77, sd 5.96, range 0.77 to 26.52
    runs with at least one hub-hub coupling: 29 of 30
weight ratio W_hp / W_pp: median 1.66 [1.55, 1.95], mean 1.99, sd 0.97, range 0.78 to 4.49
cascade reach from the strongest hub, 5 rounds: median 2.7 [2.4

## Step 4. Where the cascade asymmetry of paper 1 comes from

In [8]:
print('cascade asymmetry, hub over peripheral, five rounds, on the regenerated ensembles (5 realisations each)')
print('ensemble      allocation rule (Eq. 5)   per-link rule')
for kind in KINDS:
    va, vp = [], []
    for s in range(SEED0, SEED0 + 5):
        A = nx.to_numpy_array(make_ensemble_graph(kind, s), dtype=bool); W = A.astype(float)
        va.append(cascade_asymmetry(A, W, np.zeros(N), rngc)[2])
        vp.append(cascade_asymmetry(A, W, np.zeros(N), rngc, per_link=True)[2])
    print(f'{kind:<12}  {np.mean(va):.2f}                     {np.mean(vp):.2f}')
vp = [cascade_asymmetry(m.A, m.W, m.beta, rngc, per_link=True)[2] for m in RUNS['both']]
print(f'learned structures, full model, per-link rule: median asymmetry {np.median(vp):.2f}')
print(f'paper 1, its own cascade model: scale-free {PUB1["dhp_BA"]}, random {PUB1["dhp_ER"]}, small-world {PUB1["dhp_WS"]}')

class PerLinkModel(ElementModel):
    def spread(self, u, a):
        p, rng, N_ = self.p, self.rng, self.N
        act = rng.random(N_) < (1.0 - (1.0 - p['pn']) * np.exp(-p['gamma'] * u))
        frontier = act.copy(); rounds = 0
        while frontier.any() and rounds < 20:
            d = p['lam'] * (frontier.astype(float) @ self.A.astype(float))
            prob = p['pl'] * logistic(d, p['g'], p['theta'])
            new = (rng.random(N_) < prob) & ~act
            act |= new; frontier = new; rounds += 1
        self.n_act, self.n_rounds = float(act.sum()), float(rounds)
        return act

PLS = seeds(10)
print(f'per-link transmission as a variant of the whole model, p_l = 0.3, {len(PLS)} seeds, medians')
print('condition     compression   kappa   assortativity   essential share   activated share')
PL_VAR = {}
for name, kw in CONDS.items():
    out = []
    for sd in PLS:
        m = PerLinkModel(seed=sd, **kw)
        for _ in range(T_POP):
            m.step()
        mm = m.measures(adj=True)
        out.append((CEIL - mm['H_str'], mm['kappa'], mm['r_deg'], mm['share_E'], mm['act']))
    o = np.array(out); PL_VAR[name] = o
    print(f'{name:<12}  {np.median(o[:, 0]):.3f}         {np.median(o[:, 1]):5.2f}   {np.median(o[:, 2]):+.3f}          '
          f'{np.median(o[:, 3]):.3f}             {np.median(o[:, 4]):.2f}')
print('combined condition most concentrated on the essential elements under the per-link rule:',
      bool(np.median(PL_VAR['both'][:, 3]) > max(np.median(PL_VAR[k][:, 3]) for k in ['baseline', 'explore', 'imitate'])))

cascade asymmetry, hub over peripheral, five rounds, on the regenerated ensembles (5 realisations each)
ensemble      allocation rule (Eq. 5)   per-link rule
lattice       0.95                     0.98
small-world   1.00                     1.20
random        0.77                     1.51
scale-free    0.73                     1.97
learned structures, full model, per-link rule: median asymmetry 2.21
paper 1, its own cascade model: scale-free 3.6, random 2.25, small-world 1.46
per-link transmission as a variant of the whole model, p_l = 0.3, 10 seeds, medians
condition     compression   kappa   assortativity   essential share   activated share
baseline      0.781          9.78   +0.005          0.175             0.43
explore       0.975         10.66   -0.043          0.137             0.45
imitate       0.787          9.78   +0.051          0.192             0.41
both          0.616         12.66   -0.230          0.240             0.52
combined condition most concentrated on the essen

## Step 5. The order-of-change test that paper 2 specified in advance

In [9]:
def activation_similarity(X, every=25):
    out = []
    for t0 in range(0, X.shape[0], every):
        w = X[t0:t0 + every].astype(float)
        keep = w.std(0) > 0
        if keep.sum() < 2:
            out.append(np.nan); continue
        c = np.corrcoef(w[:, keep].T)
        iu = np.triu_indices(keep.sum(), 1)
        out.append(np.nanmean(c[iu]))
    return np.array(out)

def smooth(y, w=9):
    # centered moving average over w recording intervals, ignoring missing values
    y = np.asarray(y, float); out = np.full_like(y, np.nan); h = w // 2
    for i in range(len(y)):
        seg = y[max(0, i - h):i + h + 1]; seg = seg[~np.isnan(seg)]
        if len(seg): out[i] = seg.mean()
    return out

def normalized(y):
    # the smoothed series as a fraction of its total change, start 0, end 1
    y = smooth(y); ok = ~np.isnan(y)
    y0 = y[ok][:3].mean(); y1 = y[ok][-6:].mean()
    if abs(y1 - y0) < 1e-9:
        return np.full_like(y, np.nan)
    return (y - y0) / (y1 - y0)

def onset(y):
    z = normalized(y); ok = ~np.isnan(z)
    if not ok.any():
        return np.nan
    idx = np.flatnonzero(ok & (z >= 0.5))
    return idx[0] if idx.size else np.nan

LEADS, CURVES = [], []
for rec, X in zip(RECS['both'], XS['both']):
    C = 1 - rec['H_str'] / CEIL
    S = activation_similarity(X)
    n = min(len(C), len(S)); C, S = C[:n], S[:n]
    oc, os_ = onset(C), onset(S)
    LEADS.append(os_ - oc); CURVES.append((rec['t'][:n], C, S))
LEADS = np.array(LEADS, float)
ok = ~np.isnan(LEADS)
print(f'order of change over {int(ok.sum())} runs with a defined onset for both signals (of {len(LEADS)}); '
      f'onset = first half-crossing of each series after a centered moving average over 9 intervals (225 steps)')
print(f'onset of activation similarity minus onset of coupling concentration, in recording intervals of 25 steps: '
      f'median {np.median(LEADS[ok]):+.1f}, range {LEADS[ok].min():+.0f} to {LEADS[ok].max():+.0f}')
print(f'coupling concentration leads in {int((LEADS[ok] > 0).sum())} runs, similarity leads in {int((LEADS[ok] < 0).sum())}, '
      f'tie in {int((LEADS[ok] == 0).sum())}')
t0, C0, S0 = CURVES[0]
print(f'seed 0: coupling concentration rises from {C0[0]:.3f} to {C0[-1]:.3f}, activation similarity from {np.nanmean(S0[:3]):.3f} to {np.nanmean(S0[-6:]):.3f}')

order of change over 30 runs with a defined onset for both signals (of 30); onset = first half-crossing of each series after a centered moving average over 9 intervals (225 steps)
onset of activation similarity minus onset of coupling concentration, in recording intervals of 25 steps: median -29.0, range -138 to +13
coupling concentration leads in 4 runs, similarity leads in 26, tie in 0
seed 0: coupling concentration rises from 0.000 to 0.184, activation similarity from 0.022 to 0.002


## Step 6. Figure S4. From the lattice of paper 2 toward the region of paper 1

### The drawing style of the SI figures

In [10]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

SI_RC = {'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.labelsize': 6.6, 'xtick.labelsize': 6.0,
         'ytick.labelsize': 6.0, 'legend.fontsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
         'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 0, 'legend.frameon': False,
         'axes.spines.top': True, 'axes.spines.right': True, 'mathtext.fontset': 'dejavusans'}
INK, NEUTRAL = '#1A1A1A', '#37474F'
BLUE, RED, GOLD, GREEN, GREY, NAVY = '#2E5090', '#A4161A', '#C9A227', '#2E7D4E', '#8A8A8A', '#1F3F7A'
SI_COND = {'baseline': '#C4C9CD', 'explore': '#8A8A8A', 'imitate': '#5F6B72', 'both': NAVY}
SI_LABEL = {'baseline': 'baseline', 'explore': 'exploration', 'imitate': 'imitation', 'both': 'both'}
_SI_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_SI_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP3 = LinearSegmentedColormap.from_list('hot_div', np.vstack([_SI_COOL, _SI_HOT]))
HOT3 = LinearSegmentedColormap.from_list('hot_half', CMAP3(np.linspace(0.50, 0.92, 256)))
SI_VLIM = 2.7157337518864986                       # the scale of Fig. 2 and of ref. 3, in bits
SI_CNORM = Normalize(-SI_VLIM, SI_VLIM)
DOT = (0, (1.2, 1.4))


def si_share_colour(w):
    # an element colored by its share against an even share, as in Figs. 2 and 3
    w = np.asarray(w, float); b = np.log2(w / w.sum() * len(w))
    return CMAP3(SI_CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))


def si_figure(w, h, label):
    plt.rcParams.update(SI_RC)
    fig = plt.figure(figsize=(w, h)); fig.set_label(label)
    return fig


def si_axes(fig, l, b, w, h):
    W, H = fig.get_size_inches()
    return fig.add_axes([l / W, b / H, w / W, h / H])


def si_head(fig, ax, letter, title, dy=0.12):
    # the title centered over the plot, the letter in bold beside its left end
    W, H = fig.get_size_inches()
    bb = ax.get_position()
    y = bb.y1 + dy / H
    t = fig.text((bb.x0 + bb.x1) / 2, y, title, fontsize=7.6, ha='center', va='bottom')
    ext = t.get_window_extent(fig.canvas.get_renderer()).transformed(fig.transFigure.inverted())
    fig.text(ext.x0 - 0.06 / W, y, letter, fontsize=9.0, fontweight='bold', ha='right', va='bottom')


def si_ticks(lo, hi):
    return [lo, (lo + hi) / 2, hi]


def si_nice(lo, hi, zero=False):
    # three round ticks, minimum, middle and maximum, as tight as round numbers allow around the data
    lo, hi = (min(0.0, lo), max(0.0, hi)) if zero else (lo, hi)
    span = (hi - lo) or abs(hi) or 1.0
    for e in range(int(np.floor(np.log10(span))) - 2, int(np.floor(np.log10(span))) + 2):
        for m in (1, 1.5, 2, 2.5, 3, 4, 5, 6, 8):
            u = m * 10.0 ** e
            a = np.floor(lo / u + 1e-9) * u
            b = np.ceil(hi / u - 1e-9) * u
            n = int(round((b - a) / u))
            if n % 2:
                b += u; n += 1
            if n <= 2:
                return [round(a, 10), round(a + u, 10), round(b, 10)]
    return [lo, (lo + hi) / 2, hi]


def si_frame(ax, xt=None, yt=None, xlog=False, ylog=False, xfmt=None, yfmt=None, grid=True):
    # a line above and below, none at the sides, ticks at the given values, a dotted rule at the middle tick
    if xlog:
        ax.set_xscale('log')
    if ylog:
        ax.set_yscale('log')
    g = lambda v, _: f'{v:,g}'
    if xt is not None:
        ax.set_xlim(xt[0], xt[-1]); ax.set_xticks(xt)
        ax.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(xfmt or g))
        ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    if yt is not None:
        ax.set_ylim(yt[0], yt[-1]); ax.set_yticks(yt)
        ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(yfmt or g))
        ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
        if grid:
            for v in yt[1:-1]:
                ax.axhline(v, color='gray', alpha=0.45, ls=':', lw=0.5, zorder=0)
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)
    ax.set_axisbelow(True)


def si_cbar(fig, l, b, w, h, cmap, lo, hi, label, fmt='{:g}'):
    ax = si_axes(fig, l, b, w, h)
    ax.imshow(np.linspace(1, 0, 128)[:, None], cmap=cmap, aspect='auto', extent=[0, 1, lo, hi])
    ax.set_xticks([]); ax.set_yticks([lo, hi]); ax.set_yticklabels([fmt.format(lo), fmt.format(hi)])
    ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    for s_ in ax.spines.values():
        s_.set_visible(True); s_.set_linewidth(0.5); s_.set_linestyle(DOT)
    ax.tick_params(axis='y', length=1.6, pad=1.0, labelsize=5.6)
    ax.set_ylabel(label, fontsize=6.0, labelpad=1.5)
    return ax


def si_bars(ax, groups, colors, labels, dots=True):
    # bars at the median of each group with every run as a dot beside it
    for i, v in enumerate(groups):
        v = np.asarray(v, float)
        ax.bar(i, np.median(v), color=colors[i], width=0.58, zorder=2)
        if dots:
            ax.plot(np.full(len(v), i) + 0.20 + np.linspace(-0.05, 0.05, len(v)), v, '.', ms=1.8, color=INK,
                    alpha=0.6, zorder=3)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(labels)
    ax.set_xlim(-0.6, len(groups) - 0.4)


def si_bracket(ax, i, j, y, h, text):
    ax.plot([i, i, j, j], [y, y + h, y + h, y], '-', lw=0.6, color=INK)
    ax.text((i + j) / 2, y + h, text, ha='center', va='bottom', fontsize=6.4)


def si_radial(A, n_min=2, dmax=None):
    # elements at the angle of their place on the ring, drawn inward by degree, as in Fig. 5C
    d = A.sum(1).astype(float); n = len(d)
    th = np.pi / 2 - 2 * np.pi * np.arange(n) / n
    dmax = max(d.max() if dmax is None else dmax, n_min + 1)
    r = np.clip(np.log(dmax / np.maximum(d, n_min)) / np.log(dmax / n_min), 0, 1)
    return np.c_[r * np.cos(th), r * np.sin(th)], d


def si_structure(ax, A, W=None, isE=None, s=None, lim=1.12, dmax=None):
    # a learned structure: couplings darker and wider with weight, elements colored by their share of the
    # coupling against an even share and sized by degree, the task-essential elements ringed
    from matplotlib.collections import LineCollection
    xy, d = si_radial(A, dmax=dmax)
    iu = np.array(np.where(np.triu(A))).T
    if W is not None:
        s = (W * A).sum(1) if s is None else s
        ww = W[iu[:, 0], iu[:, 1]]; x = np.sqrt((ww - ww.min()) / max(ww.max() - ww.min(), 1e-12))
    else:
        x = np.full(len(iu), 0.3)
    o = np.argsort(x, kind='stable')
    ax.add_collection(LineCollection(xy[iu[o]], colors=plt.get_cmap('Greys')(0.25 + 0.55 * x[o]),
                                     linewidths=0.12 + 0.8 * x[o], alpha=0.75, zorder=1))
    size = 3.0 + 110.0 * ((d - 1) / (len(d) - 2)) ** 1.15
    o = np.argsort(d, kind='stable')
    ax.scatter(xy[o, 0], xy[o, 1], s=size[o], c=si_share_colour(s)[o], edgecolors=NEUTRAL, linewidths=0.25, zorder=3)
    if isE is not None:
        e = np.where(isE)[0]
        ax.scatter(xy[e, 0], xy[e, 1], s=(np.sqrt(size[e]) + 2.6) ** 2, facecolors='none', edgecolors=INK,
                   linewidths=0.6, zorder=4)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect('equal'); ax.axis('off')

In [11]:
def band(ax, recs, key, color, label, transform=lambda v: v):
    t = recs[0]['t']
    Y = np.array([transform(r[key]) for r in recs])
    med = np.median(Y, 0); lo, hi = np.percentile(Y, [25, 75], axis=0)
    ax.fill_between(t, lo, hi, color=color, alpha=0.18, lw=0)
    ax.plot(t, med, '-', lw=1.1, color=color, label=label)
    return hi.max()

fig = si_figure(6.93, 3.30, 'SI Fig S4')
XT = [0, T_POP // 2, T_POP]
ENS = {'small-world': CMAP3(0.18), 'random': CMAP3(0.72), 'scale-free': CMAP3(0.88)}

ax = si_axes(fig, 0.42, 1.95, 2.55, 0.98)
top = max(band(ax, RECS['both'], 'H_deg', NAVY, 'exploration and imitation', lambda v: CEIL - v),
          band(ax, RECS['baseline'], 'H_deg', GREY, 'baseline', lambda v: CEIL - v))
for kind in ('small-world', 'random', 'scale-free'):
    ax.axhline(REGEN[kind]['dH_deg'], ls=':', lw=0.8, color=ENS[kind])
    ax.text(T_POP * 1.01, REGEN[kind]['dH_deg'], kind, fontsize=5.6, color=ENS[kind], va='center', ha='left')
si_frame(ax, XT, si_nice(0, top, zero=True), grid=False)
ax.set_xlim(0, T_POP)
ax.legend(loc='upper left', handlelength=1.4, labelspacing=0.25)
ax.set_xlabel('step', labelpad=1); ax.set_ylabel(r'bits saved, degree shares', labelpad=1)
si_head(fig, ax, 'A', 'What the structure saves, as in the second paper')

ax = si_axes(fig, 3.95, 1.95, 2.55, 0.98)
top = max(band(ax, RECS['both'], 'kappa', NAVY, 'exploration and imitation'),
          band(ax, RECS['baseline'], 'kappa', GREY, 'baseline'))
for kind in ('small-world', 'random', 'scale-free'):
    ax.axhline(REGEN[kind]['kappa'], ls=':', lw=0.8, color=ENS[kind])
    ax.text(T_POP * 1.01, REGEN[kind]['kappa'], kind, fontsize=5.6, color=ENS[kind],
            va='top' if kind == 'small-world' else 'bottom' if kind == 'random' else 'center', ha='left')
si_frame(ax, XT, si_nice(0, top, zero=True), grid=False)
ax.set_xlim(0, T_POP)
ax.set_xlabel('step', labelpad=1); ax.set_ylabel(r'degree heterogeneity $\kappa$', labelpad=1)
si_head(fig, ax, 'B', 'The topology, as in the first paper')

SN = SNAPS['both'][0]; ts = sorted(SN)
DMAX = max(SN[tt][0].sum(1).max() for tt in ts)
W0 = 1.05
for j, tt in enumerate(ts):
    A_s, s_s, isE_s = SN[tt]
    ax = si_axes(fig, 0.95 + j * 1.85, 0.30, W0, W0)
    si_structure(ax, A_s, None, isE_s, s=s_s, dmax=DMAX)
    deg_t = A_s.sum(1)
    ax.text(0.5, -0.06, f'step {tt + 1:,}\n{CEIL - control_entropy(s_s):.2f} bits, '
            rf'$\kappa$ = {(deg_t ** 2).mean() / deg_t.mean():.1f}', transform=ax.transAxes, fontsize=5.8,
            ha='center', va='top')
    if j == 0:
        fig.text(0.5, (0.30 + W0 + 0.14) / 3.30, 'The structure along the run of the full model from seed 0',
                 fontsize=7.6, ha='center', va='bottom')
        fig.text(0.5 - 2.10 / 6.93, (0.30 + W0 + 0.14) / 3.30, 'C', fontsize=9.0, fontweight='bold', ha='right',
                 va='bottom')
plt.show()

## Step 7. The same learner on every measure (Fig. S3)

In [12]:
one, rec = RUNS['both'][0], RECS['both'][0]
dH = CEIL - rec['H_str']
mm = one.measures(adj=True)
print(f'single run: start H = {CEIL:.4f} bits')
print(f'single run end: H = {CEIL - dH[-1]:.4f} bits, compression = {dH[-1]:.3f} bits')
print(f'single run end: effective count 2^H = {2 ** (CEIL - dH[-1]):.1f} of {N}')
print(f'single run end: kappa = {mm["kappa"]:.2f}, degree assortativity r = {mm["r_deg"]:+.3f}')
print(f'single run end: mean preference = {mm["beta_mean"]:.2f}, mean commitment = {mm["f_mean"]:.3f}, '
      f'hub alignment = {mm["hub_align"]:.2f}')
print(f'single run end: share held by essential elements = {mm["share_E"]:.3f}, '
      f'mean activation = {mm["act"]:.2f}')
print(f'single run: eigenvector weighting as in the control-entropy paper = {CEIL - mm["H_adj"]:.3f} bits, '
      f'degree weighting = {CEIL - mm["H_deg"]:.3f} bits')

fig = si_figure(6.93, 3.25, 'SI Fig S3')
PW, PH = 1.18, 0.86
XS = [0.40, 2.12, 3.84, 5.56]
YS = [2.02, 0.42]
t = rec['t']; XT = [0, t[-1] // 2, t[-1]]

def panel(i, j):
    return si_axes(fig, XS[j], YS[i], PW, PH)

ax = panel(0, 0)
ax.plot(t, dH, color=RED, lw=0.9, label='strength shares')
ax.plot(t, CEIL - rec['H_deg'], color=NAVY, lw=0.9, label='degree shares')
ax.plot(t, CEIL - rec['H_adj'], color=GREEN, lw=0.9, label='eigenvector')
ax.axhline(REGEN['scale-free']['dH_deg'], ls=':', color=INK, lw=0.7)
si_frame(ax, XT, si_nice(0, max(dH.max(), (CEIL - rec['H_deg']).max()), zero=True), grid=False)
ax.text(t[-1], REGEN['scale-free']['dH_deg'] + 0.04, 'scale-free', fontsize=5.6, ha='right', va='bottom', color=NEUTRAL)
ax.legend(loc='upper left', handlelength=1.2, labelspacing=0.2, borderaxespad=0.1, fontsize=5.4)
ax.set_xlabel('step', labelpad=1); ax.set_ylabel('bits saved', labelpad=1)
si_head(fig, ax, 'A', 'Compression')

ax = panel(0, 1)
sd = np.sqrt(rec['beta_var'])
ax.fill_between(t, rec['beta_mean'] - sd, rec['beta_mean'] + sd, color=GREEN, alpha=0.18, lw=0)
ax.plot(t, rec['beta_mean'], color=GREEN, lw=0.9)
si_frame(ax, XT, si_nice(0, (rec['beta_mean'] + sd).max(), zero=True))
ax.set_xlabel('step', labelpad=1); ax.set_ylabel(r'direction preference $\bar{\beta}$', labelpad=1)
si_head(fig, ax, 'B', 'Direction')

ax = panel(0, 2)
ax.plot(t, rec['f_mean'], color=BLUE, lw=0.9)
si_frame(ax, XT, si_nice(0, rec['f_mean'].max(), zero=True))
ax.set_xlabel('step', labelpad=1); ax.set_ylabel(r'commitment $\bar{f}$', labelpad=1)
si_head(fig, ax, 'C', 'Commitment')

ax = panel(0, 3)
ax.plot(t, rec['kappa'], color=NAVY, lw=0.9)
ax.axhline(REGEN['scale-free']['kappa'], ls=':', color=INK, lw=0.7)
si_frame(ax, XT, si_nice(0, rec['kappa'].max(), zero=True), grid=False)
ax.text(t[-1], REGEN['scale-free']['kappa'] - 0.6, 'scale-free', fontsize=5.6, ha='right', va='top', color=NEUTRAL)
ax.set_xlabel('step', labelpad=1); ax.set_ylabel(r'degree heterogeneity $\kappa$', labelpad=1)
si_head(fig, ax, 'D', 'Hubs')

ax = panel(1, 0)
ax.plot(t, rec['act'], color='#AEB6BC', lw=0.6, label='activated per step')
ax.plot(t, rec['share_E'], color=NAVY, lw=0.9, label='coupling on the essential set')
si_frame(ax, XT, [0, 0.4, 0.8])
ax.legend(loc='upper right', handlelength=1.2, labelspacing=0.2, borderaxespad=0.1, fontsize=5.4)
ax.set_xlabel('step', labelpad=1); ax.set_ylabel('share', labelpad=1)
si_head(fig, ax, 'E', 'Where coupling and activity go')

ax = panel(1, 1)
s_end = np.sort(one.strengths())[::-1]; sh = s_end / s_end.sum()
ax.plot(np.arange(1, N + 1), sh, color=RED, lw=0.9)
ax.axhline(1 / N, ls=':', color=INK, lw=0.7)
ax.text(N, 1 / N * 1.15, 'even share', fontsize=5.6, ha='right', va='bottom', color=NEUTRAL)
si_frame(ax, [1, 50, 100], [1e-3, 1e-2, 1e-1], ylog=True, yfmt=lambda v, _: f'{v:g}', grid=False)
ax.set_xlabel('rank', labelpad=1); ax.set_ylabel('coupling share', labelpad=1)
si_head(fig, ax, 'F', 'Shares at step 5,000')

ax = panel(1, 2)
deg = np.sort(one.A.sum(1)); cc = 1 - np.arange(N) / N
ax.plot(deg, cc, '-', color=NAVY, lw=0.9)
ax.axvline(6, ls=':', color=GREEN, lw=0.8)
si_frame(ax, [1, 10, 100], [1e-2, 1e-1, 1], xlog=True, ylog=True, xfmt=lambda v, _: f'{v:g}',
         yfmt=lambda v, _: f'{v:g}')
ax.text(6.4, 0.012, 'lattice', fontsize=5.6, ha='left', va='bottom', color=GREEN)
ax.set_xlabel(r'degree $k$', labelpad=1); ax.set_ylabel(r'$P(\geq k)$', labelpad=1)
si_head(fig, ax, 'G', 'Degrees at step 5,000')

ax = si_axes(fig, XS[3] - 0.08, YS[1] - 0.16, PW + 0.18, PW + 0.18)
si_structure(ax, one.A, one.W, one.isE)
si_head(fig, ax, 'H', 'Structure at step 5,000', dy=-0.02)
plt.show()

single run: start H = 6.6439 bits
single run end: H = 5.3822 bits, compression = 1.262 bits
single run end: effective count 2^H = 41.7 of 100
single run end: kappa = 25.42, degree assortativity r = -0.348
single run end: mean preference = 2.62, mean commitment = 0.040, hub alignment = 1.00
single run end: share held by essential elements = 0.438, mean activation = 0.09
single run: eigenvector weighting as in the control-entropy paper = 0.226 bits, degree weighting = 0.824 bits
